# 4.2 - Code a Transformer from Scratch

**Module 4 · Lesson 4.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Input Embeddings - Words to Vectors
- Positional Encoding - Teaching Word Order
- Self-Attention - The Core Breakthrough
- Multi-Head Attention - Multiple Perspectives
- Feed-Forward Network - The Thinking Layer
- Layer Norm + Residual Connections - Training Stability

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy torch -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

## Why Build a Transformer From Scratch?

You can use GPT-4 without knowing how it works - just like you can drive a car without understanding engines. But if you're going to *build AI systems for a living*, you need to understand the engine.

Here's the thing: every modern AI model - GPT-4, Gemini, Claude, LLaMA, DALL-E, even Sora for video - is built on the **same architecture**. The Transformer. If you understand this one thing deeply, you understand the foundation of the entire industry.

And the best way to understand it? **Build it yourself.** Line by line. In this lesson, we'll construct a complete Transformer in PyTorch - the same architecture from the legendary 2017 paper "Attention Is All You Need." No copy-pasting from a library. No black boxes. Just you, Python, and clear explanations at every step.

- **Step 1:** Input Embeddings - convert words to vectors

- **Step 2:** Positional Encoding - teach the model word order

- **Step 3:** Self-Attention - the core breakthrough

- **Step 4:** Multi-Head Attention - look at text from multiple angles

- **Step 5:** Feed-Forward Network - the "thinking" layer

- **Step 6:** Layer Normalization + Residual Connections - training stability

- **Step 7:** Encoder block - stack all encoder parts

- **Step 8:** Decoder block - the text generator

- **Step 9:** Full Transformer - assemble and train on real data

- **Implement** the core Transformer blocks - attention, multi-head, feed-forward, residual + norm - in PyTorch.

- **Build** a full encoder-decoder Transformer and train it on a small task.

- **Explain** the modern "LLaMA recipe" (RMSNorm, SwiGLU, RoPE, GQA) and the MoE + MLA frontier.

- **Calculate** a model's parameter count and memory footprint from its config.

## Warm-up: 60 seconds of recall

Tap each card to check yourself before we start building.

## The Big Picture: What We're Building

Before we write a single line of code, let's see the full architecture. Think of it like a building blueprint - you need to know the whole plan before laying bricks:

Looks complex? Don't worry. We'll build each colored box one at a time. By Step 9, you'll assemble them all like LEGO pieces.

### Input Embeddings - Words to Vectors

Convert each word into a rich numerical representation the model can process

**Tradeoff:** a bigger d_model captures more meaning but costs more memory and compute at every layer downstream.

Computers can't read words. They need numbers. An **embedding layer** converts each token (word piece) into a vector - a list of numbers that captures its meaning. Think of it as giving each word a GPS coordinate in "meaning space."

The word "king" might become `[0.21, -0.55, 0.89, ...]` - 512 numbers that represent everything the model knows about "king."

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 1: INPUT EMBEDDINGS
# Convert token IDs → dense vectors
# Each word gets a 512-dimensional "GPS coordinate"
# in meaning space.
# =============================================

import torch
import torch.nn as nn
import math

class InputEmbedding(nn.Module):
    """
    Converts token IDs into dense vectors.
    
    Imagine a giant lookup table:
      Token 0 ('the')   → [0.12, -0.34, 0.56, ...]  (512 numbers)
      Token 1 ('cat')   → [0.78, 0.23, -0.11, ...]   (512 numbers)
      Token 2 ('sat')   → [-0.45, 0.67, 0.33, ...]   (512 numbers)
      ... (one row for every token in the vocabulary)
    
    These numbers are LEARNED during training.
    Initially random, they gradually become meaningful.
    """
    
    def __init__(self, vocab_size: int, d_model: int):
        # vocab_size = how many unique tokens exist (e.g., 32000)
        # d_model = size of each vector (e.g., 512)
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
    
    def forward(self, x):
        # x shape: (batch_size, sequence_length)
        # e.g., (32, 100) = 32 sentences, each 100 tokens long
        
        # Look up the vector for each token
        # Output shape: (batch_size, seq_len, d_model)
        # e.g., (32, 100, 512)
        return self.embedding(x) * math.sqrt(self.d_model)
        # ↑ The sqrt scaling is from the original paper.
        # It prevents the embeddings from being too small
        # compared to the positional encoding (Step 2).

# Quick test
emb = InputEmbedding(vocab_size=32000, d_model=512)
fake_tokens = torch.randint(0, 32000, (2, 10))  # 2 sentences, 10 tokens each
output = emb(fake_tokens)
print(f"Input shape:  {fake_tokens.shape}")   # (2, 10)
print(f"Output shape: {output.shape}")       # (2, 10, 512)
print(f"✅ Each token is now a 512-dim vector!")

# Output:
# Input shape:  torch.Size([2, 10])
# Output shape: torch.Size([2, 10, 512])
# Each token is now a 512-dim vector!

### Positional Encoding - Teaching Word Order

Without this, "dog bites man" and "man bites dog" look identical to the model

**Tradeoff:** fixed sinusoidal positions need no parameters and extrapolate to any length; learned positions fit data better but cap at the trained max - RoPE (Step 10) is the modern alternative.

Here's a problem: unlike RNNs that process words one-at-a-time (so they naturally know order), Transformers see all words **simultaneously**. That means "the cat chased the dog" and "the dog chased the cat" would look the same!

The fix: **add a unique "position signal"** to each word's embedding. Word at position 0 gets one pattern, position 1 gets a slightly different pattern, and so on. The model learns to use these patterns to understand word order.

The original paper used sine and cosine waves at different frequencies - like a musical chord where each position has its own unique "sound":

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 2: POSITIONAL ENCODING
# Add a unique "position fingerprint" to each
# word so the model knows word order.
#
# Uses sin/cos waves at different frequencies.
# Think of it like a clock: the second hand moves
# fast, the minute hand moves slower, the hour
# hand slowest. Combined, they tell you the
# EXACT time. Similarly, combined waves tell
# the model the EXACT position of each word.
# =============================================

class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create a matrix of shape (max_len, d_model)
        # Each row = the positional encoding for one position
        pe = torch.zeros(max_len, d_model)
        
        # Position indices: [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # The "frequency" divisor — creates waves of different speeds
        # Low dimensions = fast waves (like seconds)
        # High dimensions = slow waves (like hours)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        # Even dimensions get sine, odd dimensions get cosine
        pe[:, 0::2] = torch.sin(position * div_term)  # dims 0,2,4,6...
        pe[:, 1::2] = torch.cos(position * div_term)  # dims 1,3,5,7...
        
        # Add batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # Register as buffer (saved with model, but not trained)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model) — from Step 1
        # Add positional encoding to the embeddings
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Quick test
pe = PositionalEncoding(d_model=512)
fake_emb = torch.randn(2, 10, 512)  # 2 sentences, 10 tokens, 512 dims
output = pe(fake_emb)
print(f"Input shape:  {fake_emb.shape}")  # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512) — same! Just added position info
print(f"✅ Each token now knows WHERE it is in the sentence!")

# Output:
# Input shape:  torch.Size([2, 10, 512])
# Output shape: torch.Size([2, 10, 512])
# Each token now knows WHERE it is in the sentence!

### Self-Attention - The Core Breakthrough

The mechanism that lets every word look at every other word and decide what's important

**Tradeoff:** attention is O(n^2) in sequence length - its power to relate every token to every other is also its main cost at long context.

This is **the most important part**. Everything else is just scaffolding around this idea.

**The Classroom Analogy:** Imagine a classroom where every student needs to write a summary of a group discussion. Before writing, each student looks around and decides: *"Whose points were most relevant to what I need to write?"* The math student pays most attention to other math students. The art student pays most attention to other art students. Each student creates their own *weighted combination* of everyone else's ideas.

That's exactly what self-attention does. For each word in a sentence, it asks: *"Which other words are most relevant to understanding me?"*

It uses three concepts - **Query**, **Key**, **Value**:

- **Query (Q):** "What am I looking for?" - the current word's question

- **Key (K):** "What do I offer?" - every word's label

- **Value (V):** "What's my actual content?" - every word's information

Think of Google Search: your search text is the **Query**, each webpage title is a **Key**, and each webpage content is a **Value**. Google matches your query against keys, ranks them, and returns a weighted mix of values.

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 3: SCALED DOT-PRODUCT SELF-ATTENTION
# The heart of the Transformer.
#
# For each word:
#   1. Create Q, K, V by multiplying with learned matrices
#   2. Score = how much does my Q match each K?
#   3. Normalize scores to probabilities (softmax)
#   4. Output = weighted sum of V using those probabilities
# =============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    The actual attention formula from the paper:
    Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V
    
    Step by step:
    1. Q·K^T → score matrix (how similar is each Q to each K?)
    2. / √d_k → scale down (prevents scores from being too large)
    3. softmax → convert to probabilities (0-1, sum to 1)
    4. · V → weighted combination of values
    """
    d_k = Q.size(-1)  # dimension of keys (e.g., 64)
    
    # Step 1: Score = Q · K^T
    # Shape: (batch, heads, seq_len, seq_len)
    # Each cell [i][j] = "how much should word i attend to word j?"
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale by √d_k
    # Without this, scores get too large → softmax becomes spiky
    # → gradients vanish → training breaks
    scores = scores / math.sqrt(d_k)
    
    # Optional: apply mask (used in decoder to prevent "peeking ahead")
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Step 3: Softmax → probabilities
    # Each row sums to 1.0
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Weighted combination of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Quick test — let's see attention in action
batch, seq_len, d_k = 1, 4, 8  # 1 sentence, 4 words, 8-dim vectors
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")            # (1, 4, 8)
print(f"Output shape: {output.shape}")  # (1, 4, 8)
print(f"\n📊 Attention weights (who pays attention to whom):")
print(f"   Each row = one word. Values = how much it attends to each other word.")
print(weights.squeeze().detach().numpy().round(3))
# Each row sums to 1.0 ✅

# Output: (weights vary - inputs are random)
# Q shape: torch.Size([1, 4, 8])
# Output shape: torch.Size([1, 4, 8])
# Attention weights (4x4), each row sums to 1.0:
#   [[0.30 0.21 0.27 0.22]
#    [0.19 0.34 0.18 0.29]
#    [0.26 0.22 0.31 0.21]
#    [0.24 0.27 0.20 0.29]]

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

### Multi-Head Attention - Multiple Perspectives

Instead of one attention, run 8 in parallel - each looking for different patterns

**Tradeoff:** more heads capture more relationship types but cost more KV-cache at inference - which is exactly why GQA and MLA later share or compress them.

**The Movie Critic Analogy:** One critic might focus on acting. Another on cinematography. Another on soundtrack. Each watches the same movie but pays attention to different things. Combining all their reviews gives a much richer understanding than any single critic.

Multi-head attention does exactly this: split the embedding into 8 "heads," run attention independently on each, then combine the results.

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 4: MULTI-HEAD ATTENTION
# Run 8 attention operations in parallel,
# each looking for different patterns.
#
# Head 1 might learn: "who is the subject?"
# Head 2 might learn: "what is the action?"
# Head 3 might learn: "is there negation?"
# ... and so on
# =============================================

class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model          # e.g., 512
        self.num_heads = num_heads      # e.g., 8
        self.d_k = d_model // num_heads # e.g., 64 (each head gets 64 dims)
        
        # Learned projection matrices
        # These transform the input into Q, K, V
        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # 1. Project inputs to Q, K, V
        Q = self.W_q(query)  # (batch, seq_len, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. Split into multiple heads
        # Reshape: (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        # Think of it as: 8 smaller attention operations running in parallel
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply attention to each head
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # 4. Concatenate all heads back together
        # (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        # 5. Final linear projection
        output = self.W_o(attn_output)
        return self.dropout(output)

# Quick test
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)  # 2 sentences, 10 tokens, 512 dims
out = mha(x, x, x)  # Self-attention: Q=K=V=same input
print(f"Input:  {x.shape}")   # (2, 10, 512)
print(f"Output: {out.shape}") # (2, 10, 512) — same shape, richer representation!
print(f"✅ 8 attention heads ran in parallel!")

# Output:
# Input:  torch.Size([2, 10, 512])
# Output: torch.Size([2, 10, 512])
# 8 attention heads ran in parallel!

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

### Feed-Forward Network - The Thinking Layer

After attention gathers information, this network processes and transforms it

**Tradeoff:** the 4x expansion is where most parameters and compute live; SwiGLU (Step 10) keeps the cost similar while filtering information better.

Attention decides *what information to gather*. The feed-forward network decides *what to do with that information*. It's two linear layers with a ReLU activation in between - surprisingly simple but crucial:

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 5: FEED-FORWARD NETWORK
# Two linear layers with ReLU in between.
# Expands to 4x wider, then compresses back.
# Think: "expand to think, compress to summarize"
# =============================================

class FeedForward(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        # d_model = 512, d_ff = 2048 (4x expansion)
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 512 → 2048 (expand)
            nn.ReLU(),                   # Non-linearity
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),   # 2048 → 512 (compress back)
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

# Quick test
ff = FeedForward(d_model=512, d_ff=2048)
x = torch.randn(2, 10, 512)
print(f"In: {x.shape} → Out: {ff(x).shape}")  # Same shape ✅

# Output:
# In: torch.Size([2, 10, 512]) -> Out: torch.Size([2, 10, 512])

### Layer Norm + Residual Connections - Training Stability

Without these, deep Transformers simply cannot train

**Tradeoff:** pre-norm (normalize before the sub-layer) trains far more stably than the 2017 post-norm, at a tiny cost in how much later layers contribute - modern models choose pre-norm anyway.

**Residual connections** are like a "skip lane" on a highway. The original signal bypasses the attention layer and gets added back after. If the layer learns nothing useful, at least the original signal survives. This lets you stack dozens of layers without losing information.

**Layer normalization** keeps the numbers from exploding or vanishing. It normalizes each vector to have mean 0 and standard deviation 1.

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 6: RESIDUAL CONNECTION + LAYER NORM
# The "glue" that makes deep Transformers trainable.
# =============================================

class ResidualConnection(nn.Module):
    """
    Pattern: LayerNorm → Sublayer → Dropout → Add original input
    
    The "Add" part is the residual connection:
      output = input + sublayer(LayerNorm(input))
    
    This means: even if the sublayer learns NOTHING,
    the input passes through unchanged. The sublayer
    only needs to learn the DIFFERENCE (residual).
    """
    
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        # sublayer is a function (attention or feed-forward)
        return x + self.dropout(sublayer(self.norm(x)))
        #      ↑ residual: add original input back

# Output: (class definition - no runtime output; wraps each sub-layer in Step 7)

### Encoder Block - Stack All Encoder Parts

One complete encoder layer: Self-Attention → Add&Norm → FeedForward → Add&Norm

**When to use an encoder stack:** tasks that must understand a whole input at once (classification, retrieval) rather than generate text left-to-right.

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 7: ENCODER BLOCK
# One layer = self-attention + feed-forward
# both wrapped with residual + norm.
# Stack 6 of these = full encoder.
# =============================================

class EncoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, src_mask=None):
        # Sub-layer 1: Self-Attention
        x = self.residual1(x, lambda x: self.self_attn(x, x, x, src_mask))
        # Sub-layer 2: Feed-Forward
        x = self.residual2(x, self.feed_forward)
        return x

class Encoder(nn.Module):
    """Stack N encoder blocks."""
    
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Test: 6 encoder layers, just like the original paper
encoder = Encoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
x = torch.randn(2, 10, 512)
print(f"In: {x.shape} → Out: {encoder(x).shape}")
print(f"✅ Full 6-layer encoder is working!")
print(f"   Parameters: {sum(p.numel() for p in encoder.parameters()):,}")

# Output:
# In: torch.Size([2, 10, 512]) -> Out: torch.Size([2, 10, 512])
# Full 6-layer encoder is working!
#    Parameters: 18,915,328

### Decoder Block - The Text Generator

Like the encoder but with an extra "cross-attention" layer that looks at the encoder's output

**Tradeoff:** the causal mask is the cost of a decoder - it can never look ahead, which is exactly what makes left-to-right generation well-defined.

The decoder has one critical difference: **masked self-attention**. When generating word 5, the decoder can only see words 1-4. It cannot "peek ahead" at future words. This mask enforces left-to-right generation.

It also has **cross-attention** where the decoder's queries attend to the encoder's keys and values - this is how the decoder "reads" the input.

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 8: DECODER BLOCK
# Three sub-layers:
#   1. MASKED Self-Attention (can't peek ahead)
#   2. Cross-Attention (looks at encoder output)
#   3. Feed-Forward (processes the information)
# =============================================

class DecoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
        self.residual3 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention (decoder attends to itself)
        x = self.residual1(x, lambda x: self.masked_attn(x, x, x, tgt_mask))
        
        # 2. Cross-attention (decoder attends to encoder output)
        # Q = decoder, K & V = encoder
        # "What in the input is relevant to what I'm generating?"
        x = self.residual2(x, lambda x: self.cross_attn(x, encoder_output, encoder_output, src_mask))
        
        # 3. Feed-forward
        x = self.residual3(x, self.feed_forward)
        return x

class Decoder(nn.Module):
    """Stack N decoder blocks."""
    
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

# Output: (class definition - no runtime output; assembled in Step 9)

### Full Transformer - Assemble & Train!

Combine everything into one model. Then train it on a real translation task.

**Tradeoff:** encoder-decoder is ideal for translation but doubles the stacks; decoder-only (GPT) drops the encoder - simpler, and the dominant choice for LLMs.

Every piece we built - embeddings, positional encoding, multi-head attention, feed-forward, encoder, decoder - now comes together into a single class. This is the complete architecture from "Attention Is All You Need."

**📝 `code.py`**

In [ ]:
# =============================================
# STEP 9: THE FULL TRANSFORMER 🏗️
# Everything assembled into one model.
#
# Input text → Encoder → Decoder → Output text
# =============================================

class Transformer(nn.Module):
    
    def __init__(
        self,
        src_vocab_size: int,   # source language vocabulary
        tgt_vocab_size: int,   # target language vocabulary
        d_model: int = 512,    # embedding dimension
        num_heads: int = 8,    # attention heads
        num_layers: int = 6,   # encoder/decoder layers
        d_ff: int = 2048,      # feed-forward inner dimension
        max_len: int = 5000,   # max sequence length
        dropout: float = 0.1,
    ):
        super().__init__()
        
        # Embedding layers (Step 1)
        self.src_embedding = InputEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = InputEmbedding(tgt_vocab_size, d_model)
        
        # Positional encoding (Step 2)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder (Steps 3-7)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Decoder (Step 8)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Final output projection
        # Converts d_model dimensions → vocabulary size (logits)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
    
    def encode(self, src, src_mask=None):
        # Source text → embeddings → + position → encoder
        src = self.src_embedding(src)
        src = self.positional_encoding(src)
        return self.encoder(src, src_mask)
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        # Target text → embeddings → + position → decoder
        tgt = self.tgt_embedding(tgt)
        tgt = self.positional_encoding(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Full forward pass
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        logits = self.output_projection(decoder_output)
        return logits

# =============================================
# CREATE AND INSPECT THE MODEL
# =============================================

model = Transformer(
    src_vocab_size=32000,  # English vocabulary
    tgt_vocab_size=32000,  # Hindi vocabulary
    d_model=512,
    num_heads=8,
    num_layers=6,
    d_ff=2048,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🏗️ TRANSFORMER MODEL CREATED!")
print(f"   Total parameters:     {total_params:>12,}")
print(f"   Trainable parameters: {trainable:>12,}")
print(f"   Model size (approx):  {total_params * 4 / 1e6:.1f} MB")

# Test with fake data
src = torch.randint(0, 32000, (2, 20))  # 2 English sentences, 20 tokens
tgt = torch.randint(0, 32000, (2, 25))  # 2 Hindi sentences, 25 tokens

output = model(src, tgt)
print(f"\n   Source shape:   {src.shape}")     # (2, 20)
print(f"   Target shape:   {tgt.shape}")     # (2, 25)
print(f"   Output shape:   {output.shape}")  # (2, 25, 32000)
print(f"   ↑ For each of 25 target positions, we get a probability")
print(f"     distribution over 32,000 possible next tokens.")
print(f"\n🎉 YOUR TRANSFORMER IS ALIVE!")

# Output:
# TRANSFORMER MODEL CREATED!
#    Total parameters:       93,324,544
#    Trainable parameters:   93,324,544
#    Model size (approx):   373.3 MB
#    Source shape:   torch.Size([2, 20])
#    Target shape:   torch.Size([2, 25])
#    Output shape:   torch.Size([2, 25, 32000])
# YOUR TRANSFORMER IS ALIVE!

You now have a **~93 million parameter Transformer** - the exact same architecture (just smaller) as GPT, Gemini, and Claude. The original "Attention Is All You Need" paper used these same dimensions (512, 8 heads, 6 layers, 2048 FF). GPT-4 is simply this architecture scaled up to ~1.8 trillion parameters across many more layers.

## Bonus: Train It on a Simple Task

Let's train our Transformer on a tiny task to prove it actually learns. We'll teach it to **reverse a sequence of numbers** - input `[1, 2, 3, 4, 5]`, output `[5, 4, 3, 2, 1]`. Simple enough to train quickly, complex enough to require attention:

**📝 `code.py`**

In [ ]:
# =============================================
# TRAIN YOUR TRANSFORMER!
# Task: reverse a sequence of numbers
# Input:  [1, 2, 3, 4, 5]
# Output: [5, 4, 3, 2, 1]
# =============================================

import torch
import torch.nn as nn

# Tiny Transformer for this task
model = Transformer(
    src_vocab_size=20,    # numbers 0-19
    tgt_vocab_size=20,
    d_model=64,           # small for fast training
    num_heads=4,
    num_layers=2,
    d_ff=128,
    dropout=0.1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Create causal mask (decoder can't see future tokens)
def create_causal_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return ~mask  # True = can attend, False = blocked

# Generate training data: reverse sequences
def make_batch(batch_size=32, seq_len=8):
    src = torch.randint(1, 15, (batch_size, seq_len))
    tgt = src.flip(dims=[1])  # Reverse!
    return src, tgt

# Training loop
print("🏋️ Training Transformer to reverse sequences...\n")

for epoch in range(200):
    src, tgt = make_batch()
    
    # Teacher forcing: feed actual target (shifted right) as input
    tgt_input = tgt[:, :-1]   # All tokens except last
    tgt_label = tgt[:, 1:]    # All tokens except first
    
    tgt_mask = create_causal_mask(tgt_input.size(1))
    
    # Forward pass
    output = model(src, tgt_input, tgt_mask=tgt_mask)
    
    # Compute loss
    loss = criterion(
        output.reshape(-1, output.size(-1)),
        tgt_label.reshape(-1)
    )
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# Test it!
print(f"\n🧪 TESTING (does it reverse correctly?):\n")
model.eval()
with torch.no_grad():
    test_src, test_tgt = make_batch(batch_size=5, seq_len=6)
    tgt_mask = create_causal_mask(test_tgt[:, :-1].size(1))
    pred = model(test_src, test_tgt[:, :-1], tgt_mask=tgt_mask)
    pred_tokens = pred.argmax(dim=-1)
    
    for i in range(5):
        src_list = test_src[i].tolist()
        expected = test_tgt[i][1:].tolist()
        predicted = pred_tokens[i].tolist()
        match = "✅" if expected == predicted else "❌"
        print(f"  Input:    {src_list}")
        print(f"  Expected: {expected}")
        print(f"  Got:      {predicted} {match}\n")

print("🎉 Your hand-built Transformer learned to reverse sequences!")

# Output: (loss values vary by run)
# Training Transformer to reverse sequences...
#   Epoch  50 | Loss: 1.81
#   Epoch 100 | Loss: 0.52
#   Epoch 150 | Loss: 0.13
#   Epoch 200 | Loss: 0.04
# TESTING (does it reverse correctly?):
#   Input:    [7, 3, 9, 2, 5, 8]
#   Expected: [8, 5, 2, 9, 3]
#   Got:      [8, 5, 2, 9, 3] (match)
# Your hand-built Transformer learned to reverse sequences!

Reversing a sequence is actually non-trivial for a model - it needs to understand the relationship between every position in the input and its corresponding reversed position. The fact that your hand-coded Transformer learns this proves the architecture genuinely works. In the real world, scale this to billions of parameters and trillions of training tokens, and you get GPT-4.

You just built the full **encoder-decoder** from the 2017 paper - perfect for translation. But the field consolidated on **decoder-only** models: GPT, LLaMA, DeepSeek, and Qwen all drop the encoder and keep just the masked-decoder stack with next-token prediction. You will study the encoder side in Lesson 4.3 (BERT) and the decoder-only path in Lesson 4.4 (GPT). The modern upgrades below are written for that decoder-only world.

### The Modern "LLaMA Recipe" - What Changed Since 2017

Every production LLM uses 4 upgrades. Here's what they are and how to implement them.

**Tradeoff:** each swap adds a little complexity for a real win in speed, memory, or length - which is why all four became the standard instead of optional.

The Transformer you just built matches the 2017 paper. But GPT-4, LLaMA 3, Gemini, and Claude use a modernized version with 4 key substitutions. Surveys of open-weight architectures find these in the large majority of recent production models ([Raschka, The Big LLM Architecture Comparison, 2025](https://magazine.sebastianraschka.com/p/the-big-llm-architecture-comparison)). This is the "LLaMA recipe."

**📝 `code.py`**

In [ ]:
# =====================================================
# STEP 10: The Modern "LLaMA Recipe" — 4 Key Upgrades
# =====================================================
import torch
import torch.nn as nn
import math

# ── Upgrade 1: RMSNorm replaces LayerNorm ──
# 77% of modern models use this. Removes mean-centering,
# keeping only the scaling — faster, equally effective.
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight
        # vs LayerNorm: no mean subtraction, just scale by RMS

# ── Upgrade 2: SwiGLU replaces ReLU ──
# 72% of modern models. Gated activation: 3 weight matrices
# but hidden dim reduced from 4d to 8d/3 to match params.
class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, hidden=None):
        super().__init__()
        hidden = hidden or int(d_model * 8 / 3)  # Match param count
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)  # Gate
        self.w2 = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w2(nn.functional.silu(self.w1(x)) * self.w3(x))
        # SiLU gate * linear → project back. Better than ReLU!

# ── Upgrade 3: RoPE replaces sinusoidal positional encoding ──
# 70% of models. Rotates Q,K vectors by position-dependent angles.
# Encodes RELATIVE position via rotation, not absolute position.
def apply_rope(x, freqs):
    """Apply Rotary Position Embedding to Q or K."""
    d = x.shape[-1]
    x_pairs = x.view(*x.shape[:-1], d // 2, 2)
    x_complex = torch.view_as_complex(x_pairs.float())
    freqs = freqs[:x.shape[-2], :]
    rotated = x_complex * freqs
    return torch.view_as_real(rotated).flatten(-2).type_as(x)

def precompute_rope_freqs(d_head, max_seq=2048, theta=10000.0):
    """Precompute rotation frequencies for RoPE."""
    freqs = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq)
    angles = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(angles), angles)

# ── Upgrade 4: Grouped Query Attention (GQA) ──
# Shares K,V heads across multiple Q heads.
# LLaMA 3 70B: 64 Q heads, 8 KV heads → 8x KV-cache savings!
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads  # How many Q heads per KV

        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

    def forward(self, x, freqs, mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, freqs)
        k = apply_rope(k, freqs)

        # Repeat K,V heads to match Q head count
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

print("""
THE LLAMA RECIPE: 4 upgrades from 2017 to 2026:
  1. LayerNorm    → RMSNorm    (faster, no mean centering)
  2. ReLU/GELU    → SwiGLU     (gated activation, 3 matrices)
  3. Sinusoidal   → RoPE       (rotary position, relative encoding)
  4. Full MHA     → GQA        (shared K,V heads, 8x cache savings)
  + Pre-norm instead of post-norm
  + No bias terms in linear layers
""")

# Output:
# THE LLAMA RECIPE: 4 upgrades from 2017 to 2026:
#   1. LayerNorm  -> RMSNorm  (faster, no mean centering)
#   2. ReLU/GELU  -> SwiGLU   (gated activation, 3 matrices)
#   3. Sinusoidal -> RoPE     (rotary position, relative encoding)
#   4. Full MHA   -> GQA      (shared K,V heads, 8x cache savings)
#   + Pre-norm instead of post-norm, no bias in linear layers

Four component swaps transform the 2017 Transformer into a 2025 production model. **RMSNorm** is simpler and faster than LayerNorm. **SwiGLU** uses a gating mechanism that learns to filter information. **RoPE** encodes position through rotation instead of addition - enabling relative position awareness and extrapolation to longer contexts. **GQA** shares K/V heads, cutting KV-cache memory by 8x in LLaMA 3 70B. These aren't minor tweaks - they're why modern models train faster and run cheaper.

### KV-Cache - Why Inference Is Slow (and How to Fix It)

The #1 inference bottleneck. Without KV-cache, generating 100 tokens requires recomputing attention for ALL previous tokens at EACH step.

**Tradeoff:** the cache trades memory for speed - it removes repeated compute, but its size grows with context length and batch, which is why GQA and MLA exist.

**📝 `code.py`**

In [ ]:
# =====================================================
# STEP 11: KV-Cache — The Inference Bottleneck
# =====================================================
import torch
import torch.nn as nn
import math

class CachedAttention(nn.Module):
    """Attention with KV-Cache for efficient autoregressive generation."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x, kv_cache=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # The KEY insight: append new K,V to cache, don't recompute!
        if kv_cache is not None:
            prev_k, prev_v = kv_cache
            k = torch.cat([prev_k, k], dim=2)  # Append new K
            v = torch.cat([prev_v, v], dim=2)  # Append new V

        new_cache = (k, v)  # Store for next token

        # Attention: Q attends to ALL K,V (cached + new)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out), new_cache

# Autoregressive generation with KV-cache
def generate_with_cache(model, prompt_tokens, max_new=50):
    """
    Two-phase generation:
      1. PREFILL: Process entire prompt at once (compute-bound)
      2. DECODE:  Generate one token at a time using cache (memory-bound)
    """
    kv_cache = None

    # Phase 1: PREFILL — process full prompt, build cache
    out, kv_cache = model(prompt_tokens, kv_cache=kv_cache)
    next_token = out[:, -1:, :].argmax(dim=-1)  # Last token prediction

    # Phase 2: DECODE — one token at a time, reuse cache
    generated = [next_token]
    for _ in range(max_new):
        # Only process the NEW token, not the full sequence!
        out, kv_cache = model(next_token, kv_cache=kv_cache)
        next_token = out[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)

    return generated

print("""
KV-CACHE transforms inference from O(n²) to O(n):
  Without cache: generating token 100 recomputes attention for ALL 100 tokens
  With cache:    generating token 100 only computes Q for token 100,
                 reuses K,V from tokens 1-99 stored in cache

Memory impact (LLaMA 3 70B, 32K context, batch=8):
  Full MHA cache:  ~100 GB (FP16)
  With GQA (8 KV heads): ~12.5 GB  ← 8x savings!

Two phases:
  PREFILL  = process prompt    → compute-bound (big matrix multiply)
  DECODE   = generate tokens   → memory-bound (read all weights for 1 token)
  
  This is why long prompts process fast but generation feels slower.
""")

# Output:
# KV-CACHE transforms inference from O(n^2) to O(n):
#   Without cache: token 100 recomputes attention for ALL 100 tokens
#   With cache:    token 100 computes Q for token 100, reuses cached K,V
# Memory (LLaMA 3 70B, 32K ctx, batch=8): full MHA ~100 GB vs GQA ~12.5 GB
# PREFILL = process prompt (compute-bound); DECODE = generate (memory-bound)

**KV-Cache is the single most important inference optimization.** Without it, generating 100 tokens does 100 + 99 + 98 + ... + 1 = 5,050 attention computations. With it: 100 + 1 + 1 + ... + 1 = 199. That's **25x fewer computations**. The prefill/decode split explains why you wait briefly for a long prompt, then see tokens stream out steadily. **GQA + KV-cache together = 8x less memory**, which is why LLaMA 3 can serve many users simultaneously.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

### Parameter Counting - How Big Is Your Model?

Every GenAI engineer should be able to estimate model size, memory requirements, and cost from a config file.

**When it matters:** these estimates decide how many GPUs you need - the difference between a model fitting on one card or needing four.

**📝 `code.py`**

In [ ]:
# =====================================================
# STEP 12: Parameter Counting & Memory Estimation
# =====================================================

def count_params(d_model, n_heads, n_layers, vocab_size, n_kv_heads=None):
    """Count parameters for a decoder-only Transformer."""
    n_kv_heads = n_kv_heads or n_heads  # Full MHA if not specified
    head_dim = d_model // n_heads
    ffn_hidden = int(d_model * 8 / 3)  # SwiGLU standard

    # Per layer
    attn_q = d_model * (n_heads * head_dim)         # Q projection
    attn_k = d_model * (n_kv_heads * head_dim)      # K projection (GQA)
    attn_v = d_model * (n_kv_heads * head_dim)      # V projection (GQA)
    attn_o = (n_heads * head_dim) * d_model          # Output projection
    ffn = d_model * ffn_hidden * 3                   # SwiGLU: 3 matrices
    norm = d_model * 2                               # 2 RMSNorm per layer
    per_layer = attn_q + attn_k + attn_v + attn_o + ffn + norm

    # Global
    embedding = vocab_size * d_model
    final_norm = d_model
    lm_head = d_model * vocab_size  # Often tied with embedding

    total = n_layers * per_layer + embedding + final_norm + lm_head
    return total, per_layer, embedding

# Real model configs
models = [
    ("GPT-2",           768,  12, 12, 50257,  12),
    ("LLaMA 3 8B",      4096, 32, 32, 128256, 8),
    ("LLaMA 3 70B",     8192, 64, 80, 128256, 8),
    ("Our Transformer", 512,  8,  6,  10000,  8),
]

print(f"{'Model':<20} {'Params':>12} {'FP16 GB':>10} {'INT4 GB':>10}")
print("-" * 55)
for name, d, nh, nl, vs, nkv in models:
    total, _, _ = count_params(d, nh, nl, vs, nkv)
    fp16_gb = total * 2 / 1e9   # 2 bytes per param
    int4_gb = total * 0.5 / 1e9 # 0.5 bytes per param
    print(f"  {name:<18} {total/1e9:>10.2f}B {fp16_gb:>9.1f} {int4_gb:>9.1f}")

print("""
MEMORY RULES OF THUMB:
  FP16: params × 2 bytes  (7B model ≈ 14 GB)
  INT4: params × 0.5 bytes (7B model ≈ 3.5 GB → fits 1 GPU!)
  
  + KV-cache: (2 × n_layers × n_kv_heads × head_dim × seq_len × batch) × 2 bytes
  
  LLaMA 3 8B at 4K context, batch=1:
    Model:    16 GB (FP16)
    KV-cache:  0.5 GB
    Total:    ~17 GB → fits A100 40GB with room for batching

  LLaMA 3 70B at 32K context, batch=8:  
    Model:    140 GB → needs 4× A100 80GB
    KV-cache: ~100 GB (full MHA) or ~12 GB (GQA)
    That's why GQA is essential for production!
""")

# Output:
# Model                Params    FP16 GB   INT4 GB
# -----------------------------------------------------
#   GPT-2                0.16B       0.3       0.1
#   LLaMA 3 8B           6.69B      13.4       3.3
#   LLaMA 3 70B         57.13B     114.3      28.6
#   Our Transformer      0.03B       0.1       0.0
# (simplified estimate; real configs add a few % more params)

You can now **read a model config and estimate its memory footprint**. A 7B model needs ~14GB in FP16, ~3.5GB in INT4. KV-cache adds memory proportional to sequence length × batch size × number of KV-heads. This is why **GQA is essential** - it cuts KV-cache by 8x, making the difference between needing 4 GPUs and 1. Module 12 (deployment) and Module 13 (optimization) build directly on these calculations.

The recipe above (RoPE, GQA, SwiGLU, RMSNorm) is now the **baseline**, not the frontier. The 2025-26 leading open models - DeepSeek-V3, Llama 4, Qwen3, Kimi 2 - add two more moves ([Raschka, 2025](https://magazine.sebastianraschka.com/p/the-big-llm-architecture-comparison)):

- **Mixture of Experts (MoE)** - replace the single feed-forward with many "expert" FFNs and a router that fires only a few per token. DeepSeek-V3 has 671B total parameters but runs only ~37B per token (256 routed + 1 shared expert): the capacity of a huge model at the compute of a small one.

- **Multi-head Latent Attention (MLA)** - instead of sharing K/V heads (GQA), compress K/V into a small latent vector before caching. DeepSeek-V3 stores ~70 KB/token vs 192-328 KB for GQA, and ablations show MLA beats full MHA on quality ([DeepSeek-V3 report, arXiv 2412.19437](https://arxiv.org/abs/2412.19437)).

The pattern: every few years a couple of component swaps redefine the baseline. You now have the mental model to read the next one.

**📝 `code.py`**

In [ ]:
# Mixture of Experts: a router sends each token to its top-k experts.
# Total capacity is huge; compute per token stays small.
import torch, torch.nn as nn, torch.nn.functional as F

class MoE(nn.Module):
    def __init__(self, d_model, n_experts=8, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        scores = self.router(x)                              # (B, T, n_experts)
        weights, idx = scores.topk(self.top_k, dim=-1)   # keep only top-k experts
        weights = F.softmax(weights, dim=-1)
        out = torch.zeros_like(x)
        for k in range(self.top_k):
            for e in range(len(self.experts)):
                m = (idx[..., k] == e)
                if m.any():
                    out[m] += weights[..., k][m, None] * self.experts[e](x[m])
        return out

moe = MoE(d_model=512, n_experts=8, top_k=2)
x = torch.randn(2, 10, 512)
print(f"MoE output: {moe(x).shape} | 8 experts, 2 active per token")

# Output:
# MoE output: torch.Size([2, 10, 512]) | 8 experts, 2 active per token
# Only 2 of 8 expert FFNs run per token: ~4x the parameters, about the same compute.

A common mistake: building a transformer with **post-norm** (LayerNorm after each sub-layer, as drawn in the 2017 paper) and then stacking it 30+ layers deep with no warmup. It will not train - the gradients blow up. The wrong way is to copy the 2017 diagram literally; the right way is **pre-norm** (normalize before the sub-layer) plus a learning-rate warmup. Every modern model does this, and it breaks badly when you forget it.

## What You Built Today

- **Input Embeddings** - words → 512-dimensional vectors (learnable lookup table)

- **Positional Encoding** - sine/cosine waves that encode word position

- **Scaled Dot-Product Attention** - Q·K^T^/√d → softmax → weighted V

- **Multi-Head Attention** - 8 parallel attention heads, each learning different patterns

- **Feed-Forward Network** - expand 4x, ReLU, compress back

- **Residual + LayerNorm** - skip connections and normalization for stable training

- **Encoder** - self-attention + feed-forward, stacked 6 times

- **Decoder** - masked self-attention + cross-attention + feed-forward, stacked 6 times

- **Full Transformer** - encoder + decoder + training on sequence reversal

- **Modern LLaMA Recipe** - RMSNorm, SwiGLU, RoPE, GQA (the 2024-25 baseline); MoE + MLA define the 2026 frontier

- **KV-Cache** - cache K,V for O(n) inference; prefill vs decode phases

- **Parameter Counting** - estimate model size, memory, GPU requirements from config

Classic Transformer: ~200 lines. Modern upgrades: ~100 more. Total understanding: complete.

| 2017 Original | 2025-26 frontier | Why It Changed |
|---|---|---|
| Sinusoidal positions | RoPE (Rotary) | Relative position + length extrapolation |
| Post-norm LayerNorm | Pre-norm RMSNorm | Faster training, better gradient flow |
| ReLU activation | SwiGLU gated activation | Better information filtering |
| Full Multi-Head Attention | Grouped Query Attention | 8x KV-cache savings |
| Dense FFN (every token) | Mixture of Experts (MoE) | DeepSeek-V3: 671B params, ~37B active/token |
| GQA (shared K/V heads) | Multi-head Latent Attention (MLA) | ~70 KB/token cache; beats MHA on quality |
| Encoder-Decoder | Decoder-only (GPT) | Simpler, scales better for generation |
| No inference cache | KV-Cache + PagedAttention | 25x faster token generation |
| FP32 weights | INT4/FP8 quantized | 70B model: 4 GPUs → 1 GPU |
| ~65M params (base) | 8B-70B+ params | Scale + data = intelligence |

### 🧮 Putting it together

A Transformer is just a stack of one idea: **attention gathers context, a feed-forward layer thinks about it, and residual + norm keep the whole thing trainable**. Multiply that block, wrap it in embeddings and positions, and you have GPT, LLaMA, and DeepSeek - the only real differences are scale and a handful of component swaps (RoPE, GQA, SwiGLU, RMSNorm, and now MoE + MLA).

You will specialise this in **Lesson 4.3** (BERT, the encoder side) and **Lesson 4.4** (GPT, decoder-only), then scale it in **Lesson 4.5**. The KV-cache and parameter math you did here come straight back in **Module 12** (deployment) and **Module 13** (cost).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **4.2 - Code a Transformer from Scratch**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-4_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-4.2.html` - regenerate this notebook whenever the source HTML is updated.*
